# **Proyecto ML - Machine Learning**
## **Grupo 1**

### 0. Importamos las librerías necesarias

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
import ast

from sklearn.model_selection import train_test_split

pd.options.mode.copy_on_write = True
pd.set_option("display.float_format", "{:.2f}".format)

### 1. Cargamos los datos

Partimos de los datos obtenidos para el Project Break - EDA,<br>
en el que unificamos los datos de 3 trimestres en un dataset anual limpio de duplicados.

In [ ]:
df = pd.read_csv("./src/data/df_alquileres_original.csv", index_col= "id")

#### 1.1 Exploración de los datos

Mostramos su forma, información general, descripción estadística de las variables numéricas, las columnas del DataFrame y una primera vista del mismo.

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df.head(5)

### 2. Elección de target y definición de X e Y

Elegimos **estimated_revenue_l365d** como variable target, ya que es la variable que nos informa sobre la ganancia o beneficio esperado por un anuncio en todo un año.

Por tanto nuestro como nuestro objetivo es predecir cuanto dinero genera anualmente cada propiedad del dataset, haremos un modelo supervisado de regresion.


In [ ]:
y = df["estimated_revenue_l365d"]
X = df.drop("estimated_revenue_l365d", axis=1)

### 3. División en train y test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Guardamos los datos del test

X_test.to_csv("X_test.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

### 4. Análisis del target y su distribución

In [ ]:
df["estimated_revenue_l365d"].describe()

In [ ]:
plt.hist(df["estimated_revenue_l365d"], bins=50)
plt.title("Distribución del revenue anual estimado")
plt.show()

La variable objetivo presenta una distribución fuertemente sesgada a la derecha, con presencia de valores extremos muy elevados. La media es significativamente superior a la mediana, lo que confirma la asimetría. Además, existen valores 0 que deberán analizarse para determinar si corresponden a ausencia de actividad o a posibles errores.

Con el fin de reducir la asimetria, hacer la distribucion mas "normal" y reducir el impacto de los outliers vamos a hacer la función logaritmica.

In [ ]:
y_log = np.log1p(df["estimated_revenue_l365d"])

In [ ]:
plt.hist(np.log1p(df["estimated_revenue_l365d"]), bins=50)
plt.title("Distribución log-transformada del revenue")
plt.show()

### 5. Comprensión de variables 

In [ ]:
df.dtypes.value_counts()

In [ ]:
df.select_dtypes(include=["object", "string"]).shape

In [ ]:
df.columns

In [ ]:
# Hacemos una primera eliminacion de variables que no vamos a utilizar

cols_drop = [
    "listing_url",
    "scrape_id",
    "picture_url",
    "host_url",
    "host_thumbnail_url",
    "host_picture_url"
]

df = df.drop(columns=cols_drop, errors="ignore")

#### 5.1 Analisis univariante.

Dividimos las variables en numericas y categoricas.

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

print("Numéricas:", len(num_cols))
print("Categóricas:", len(cat_cols))

In [ ]:
# Analizamos las variables numericas excluyendo la target.

num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

num_cols.remove("estimated_revenue_l365d")

In [ ]:
df[num_cols].describe().T

In [ ]:
df[num_cols].hist(figsize=(20,15), bins=30)
plt.tight_layout()
plt.show()

Durante el análisis univariante se observa que muchas de las variables numéricas presentan una fuerte asimetría, con distribuciones sesgadas hacia la derecha y presencia de valores extremos. Esto sugiere la necesidad de un tratamiento posterior de outliers y posibles transformaciones.

Se identifican variables potencialmente redundantes, especialmente dentro de los bloques de disponibilidad (availability_30, availability_60, availability_90, availability_365, availability_eoy) y de restricciones de estancia (minimum_nights, maximum_nights y sus derivados como minimum_minimum_nights, maximum_maximum_nights, etc.). Estas variables miden conceptos muy similares en distintos horizontes temporales o como agregaciones estadísticas, por lo que es esperable que presenten una alta correlación entre sí. Esto puede generar multicolinealidad en modelos lineales, por lo que posteriormente se evaluará la conveniencia de seleccionar únicamente las variables más representativas de cada bloque.

Asimismo, se detectan variables que actúan como identificadores (por ejemplo, host_id), las cuales no aportan información predictiva real y deben ser eliminadas del modelado.

Por último, algunas variables como los review_scores presentan una variabilidad reducida, concentrándose en valores altos (entre 4 y 5), lo que puede limitar su capacidad explicativa. También se detectan valores extremos en varias variables que deberán analizarse y tratarse en etapas posteriores del preprocesamiento.

In [ ]:
# Eliminamos esta columna porque esta completamente vacia.

df = df.drop(columns=["calendar_updated"], errors="ignore")

In [ ]:
# Recalculo las variables numericas despues de la eliminacion de la variable anterior.

num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
num_cols.remove("estimated_revenue_l365d")

#### 5.2 Analizamos la correlacion entre las variables numericas y la variable target.


In [ ]:
# Creamos revenue_log
df["revenue_log"] = np.log1p(df["estimated_revenue_l365d"])

# Recalculamos numéricas EXCLUYENDO target y revenue_log
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

num_cols.remove("estimated_revenue_l365d")
num_cols.remove("revenue_log")

# Ahora calculamos correlaciones
corr_matrix = df[num_cols + ["revenue_log"]].corr()

corr_with_target = corr_matrix["revenue_log"].sort_values(ascending=False)

corr_with_target.head(15)

Del análisis bivariante se observa que el bloque de variables con mayor relación con la variable objetivo es el compuesto por la ocupación estimada anual (estimated_occupancy_l365d) y las métricas asociadas al volumen de reseñas (number_of_reviews_ltm, number_of_reviews_ly, number_of_reviews_l30d, reviews_per_month, number_of_reviews).

Estas variables presentan las correlaciones más elevadas con el revenue log-transformado, lo que sugiere que la intensidad de uso del alojamiento y su nivel de actividad están fuertemente vinculados con los ingresos generados.

Por otro lado, las variables estructurales del alojamiento (como capacidad o características físicas) muestran una correlación moderada, indicando que influyen en el revenue, aunque en menor medida que las variables de actividad.

Finalmente, las variables relacionadas con restricciones de estancia (bloque de “nights”) presentan una relación más débil y potencialmente redundante entre sí, por lo que será necesario evaluar su posible reducción para evitar problemas de multicolinealidad en modelos posteriores.

In [ ]:
# Vemos la correlacion entre las features mas importantes 

important_vars = [
    "estimated_occupancy_l365d",
    "number_of_reviews_ltm",
    "number_of_reviews_ly",
    "number_of_reviews_l30d",
    "reviews_per_month",
    "number_of_reviews"
]

df[important_vars].corr()

Dado el elevado grado de colinealidad detectado dentro del bloque de variables de actividad, se optará por construir distintos modelos utilizando subconjuntos representativos de estas variables, evitando incluir simultáneamente métricas altamente correlacionadas para prevenir problemas de multicolinealidad.

#### 5.3 Analisis variante de las variables categoricas

In [ ]:
len(cat_cols)

In [ ]:
for col in cat_cols:
    print(f"Para \033[1m{col}\033[0m hay {df[col].nunique()} valores únicos")

In [ ]:
cat_model = [col for col in cat_cols if df[col].nunique() <= 25]
cat_model

Tras el análisis univariante de las variables categóricas, se ha optado por seleccionar únicamente aquellas con baja o moderada cardinalidad y potencial capacidad explicativa respecto a la variable objetivo.

Se descartan variables de texto libre, fechas técnicas o identificadores, así como aquellas con excesivo número de categorías que podrían generar un alto número de variables dummy y aumentar innecesariamente la complejidad del modelo.

 Las variables seleccionadas presentan una estructura interpretable, categorías bien definidas y relevancia conceptual en el contexto del negocio (tipo de alojamiento, características del anfitrión y disponibilidad), lo que las convierte en candidatas adecuadas para el análisis bivariante y su posible inclusión en modelos predictivos posteriores.

In [ ]:
cat_analysis = [
    "room_type",
    "host_is_superhost",
    "instant_bookable",
    "host_response_time",
    "neighbourhood_group_cleansed",
    "host_identity_verified",
    "host_has_profile_pic"
]

In [ ]:
for col in cat_analysis:
    print("\n", col)
    print(df.groupby(col)["revenue_log"].mean().sort_values())

In [ ]:
import math

n_cols = 3
n_rows = math.ceil(len(cat_analysis) / n_cols)

plt.figure(figsize=(18, 5 * n_rows))

for i, col in enumerate(cat_analysis):
    ax = plt.subplot(n_rows, n_cols, i + 1)

    # Caso especial: Top 10 barrios por revenue medio
    if col == "neighbourhood_group_cleansed":
        
        top10 = (
            df.groupby(col)["revenue_log"]
              .mean()
              .sort_values(ascending=False)
              .head(10)
              .index
        )
        
        df_plot = df[df[col].isin(top10)].copy()

        # Ordenamos de mayor a menor revenue medio
        order = (
            df_plot.groupby(col)["revenue_log"]
                   .mean()
                   .sort_values(ascending=False)
                   .index
        )

        sns.boxplot(
            x=col,
            y="revenue_log",
            data=df_plot,
            order=order,
            ax=ax
        )

    else:
        sns.boxplot(
            x=col,
            y="revenue_log",
            data=df,
            ax=ax
        )

    ax.tick_params(axis='x', rotation=45)
    ax.set_title(col)

plt.tight_layout()
plt.show()

En el análisis bivariante de variables categóricas se observan diferencias claras en la distribución del revenue según ciertas categorías.

Destaca especialmente **room_type**, donde los alojamientos de tipo Entire home/apt muestran mayores valores de revenue_log frente a habitaciones privadas o compartidas. Variables relacionadas con el anfitrión como **host_is_superhost** y **host_response_time** también presentan una asociación positiva con el revenue, mientras que **instant_bookable** muestra un efecto más moderado.

Por último, **neighbourhood_group_cleansed** evidencia variaciones entre distritos, aunque su mayor número de categorías podría requerir estrategias de reducción o agrupación para su uso en modelos posteriores.

In [ ]:
for col in cat_analysis:
    print("\n", col)
    print(df[col].value_counts(dropna=False).head(15))

In [ ]:
df["host_response_time"].value_counts(dropna=False)

La variable **host_response_time** presenta un claro desbalance hacia respuestas rápidas, concentrándose principalmente en la categoría "within an hour". Además, se observa un porcentaje relevante de valores faltantes (~23%), lo que requerirá imputación o tratamiento específico. Dado que la variable posee un orden natural en sus categorías, puede considerarse su codificación ordinal para capturar mejor la relación entre rapidez de respuesta y revenue.

In [ ]:
response_order = {
    "within an hour": 0,
    "within a few hours": 1,
    "within a day": 2,
    "a few days or more": 3
}

df["host_response_time_ord"] = df["host_response_time"].map(response_order)

In [ ]:
df["room_type_simplified"] = df["room_type"].replace({
    "Shared room": "Other",
    "Hotel room": "Other"
})

In [ ]:
bin_cols = ["host_is_superhost", "host_identity_verified", "host_has_profile_pic", "instant_bookable"]

for c in bin_cols:
    if c in df.columns:
        df[c] = df[c].fillna("Unknown")

En esta fase se ha realizado un preprocesamiento específico de algunas variables categóricas con el objetivo de prepararlas para el modelado.<br>En primer lugar, la variable host_response_time, que posee un orden natural en sus categorías, ha sido transformada en una variable ordinal numérica. Esta codificación permite preservar la jerarquía implícita en el tiempo de respuesta del anfitrión (desde respuestas más rápidas hasta más lentas), facilitando que los modelos puedan capturar mejor su posible relación con el revenue.

Por otro lado, se ha simplificado la variable room_type, agrupando las categorías con muy baja representación ("Shared room" y "Hotel room") bajo una nueva categoría común ("Other"). Esta decisión reduce la dispersión de información y evita que categorías con pocos registros introduzcan ruido en el modelo.<br>Finalmente, en las variables binarias relacionadas con características del anfitrión (host_is_superhost, host_identity_verified, host_has_profile_pic e instant_bookable), se han imputado los valores faltantes con la categoría "Unknown", evitando la pérdida de observaciones y manteniendo la coherencia categórica para su posterior codificación.

### 6. Reducción o eliminación de features preliminar

In [ ]:
# Eliminamos variables de texto largas

text_cols = [
    "name",
    "description",
    "neighborhood_overview",
    "host_about",
    "amenities"
]

df = df.drop(columns=[c for c in text_cols if c in df.columns])

In [ ]:
# Eliminamos identificadores sin capacidad predictiva.

id_cols = ["host_id"]

df = df.drop(columns=[c for c in id_cols if c in df.columns])

En esta fase se realiza una depuración moderada del conjunto de variables con el objetivo de eliminar información no estructurada y posibles identificadores sin capacidad predictiva, manteniendo, no obstante, aquellas variables potencialmente redundantes que podrían aportar valor en modelos no lineales.<br>Dado que el análisis contempla tanto modelos lineales como modelos basados en árboles, se opta por una estrategia intermedia: reducir variables claramente irrelevantes sin aplicar una eliminación excesivamente restrictiva, permitiendo posteriormente comparar el comportamiento de distintos algoritmos frente a posibles problemas de multicolinealidad.

In [ ]:
# Dejamos unicamente la variable availability_365 del bloque de availability
#  ya que muestran una correlacion muy fuerte.

availability_cols = [
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_eoy"
]

df = df.drop(columns=[c for c in availability_cols if c in df.columns])

In [ ]:
# Mantenemos las variables number_of_reviews_ltm y reviews_per_month del bloque de reviews.

reviews_drop = [
    "number_of_reviews",
    "number_of_reviews_ly",
    "number_of_reviews_l30d"
]

df = df.drop(columns=[c for c in reviews_drop if c in df.columns])

In [ ]:
# Mantenemos unicamente minimum_nights y maximum_nights del bloque de nights.

nights_drop = [
    "minimum_minimum_nights",
    "minimum_maximum_nights",
    "maximum_minimum_nights",
    "maximum_maximum_nights",
    "minimum_nights_avg_ntm",
    "maximum_nights_avg_ntm"
]

df = df.drop(columns=[c for c in nights_drop if c in df.columns])


### 7.Duplicados. Comprobación y eliminación

In [ ]:
duplicates = df.duplicated().sum()
print("Número de duplicados:", duplicates)

In [ ]:
# Eliminamos los duplicados 
df = df.drop_duplicates()
print("Nuevo tamaño del dataset:", df.shape)

### 8. Detección y tratamiento de missings

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

En la revisión de valores faltantes se identifican variables con un porcentaje extremadamente elevado de datos ausentes, como license o ciertas variables geográficas de alta cardinalidad, que se eliminan por su escasa utilidad y elevada incompletitud.<br>En el resto de variables, se opta por estrategias de imputación diferenciadas según su naturaleza: imputación con mediana para variables numéricas, creación de categorías específicas para variables categóricas y tratamiento especial en variables relacionadas con reviews, donde la ausencia de datos puede reflejar la inexistencia de valoraciones.

In [ ]:
# 1) Eliminamos columnas con demasiados NaN y baja utilidad
drop_missing_cols = ["license", "host_neighbourhood", "neighbourhood", "host_location"]
df = df.drop(columns=[c for c in drop_missing_cols if c in df.columns], errors="ignore")


# 2) Eliminamos filas sin target
df = df.dropna(subset=["estimated_revenue_l365d"])


# Recalculamos revenue_log para asegurar consistencia
df["revenue_log"] = np.log1p(df["estimated_revenue_l365d"])


# 3) PRICE: convertir a numérico + imputar con mediana
if "price" in df.columns:
    df["price"] = (
        df["price"]
        .astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .str.replace(r"\s+", "", regex=True)
    )
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["price"] = df["price"].fillna(df["price"].median())


# 4) Imputación numéricas básicas (mediana)
num_impute = ["beds", "bathrooms", "bedrooms"]
for col in num_impute:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())


# 5) host_response_time_ord: -1 como categoría "Unknown"
if "host_response_time_ord" in df.columns:
    df["host_response_time_ord"] = pd.to_numeric(df["host_response_time_ord"], errors="coerce")
    df["host_response_time_ord"] = df["host_response_time_ord"].fillna(-1)


# 6) Review scores: flag + imputación con 0
if "review_scores_rating" in df.columns:
    df["has_reviews"] = df["review_scores_rating"].notnull().astype(int)

review_cols = [c for c in df.columns if "review_scores" in c]
for col in review_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(0)


# 7) Tasas del host: convertir de "%" a numérico + mediana
rate_cols = ["host_response_rate", "host_acceptance_rate"]
for col in rate_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace("%", "", regex=False)
            .str.replace(r"\s+", "", regex=True)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())


# 8) Comprobación final
missing_final = df.isnull().sum().sort_values(ascending=False)
print("Missings restantes (>0):")
print(missing_final[missing_final > 0])
print("\nTamaño final del dataset:", df.shape)

In [ ]:
# Tratamos los missings restantes

'''1) last_review es una columna con valor tipo fecha, por lo que nos interesa sacar una columna
    booleana que muestre si ha tenido reviews o no y otra que muestre los días desde la última review,
    imputando 9999 para los casos en que nunca ha habido review
'''
df["has_reviews"] = df["last_review"].notna().astype(int)
df.last_review = pd.to_datetime(df.last_review, errors="coerce")
reference_date = pd.Timestamp("2026-02-27")
df["days_since_last_review"] = (reference_date - df.last_review).dt.days
df["days_since_last_review"] = df["days_since_last_review"].fillna(9999)
df["days_since_last_review"].astype(int)

In [ ]:
'''2) first_review hace referencia a la fecha de primera review en caso de que la hubiera
    por lo que vamos a transformar a días e imputamos 0 a los nulos
'''
df.first_review = pd.to_datetime(df.first_review, errors="coerce")
df["review_lifetime"] = (df.last_review - df.first_review).dt.days
df["review_lifetime"] = df["review_lifetime"].fillna(0)
df.review_lifetime.astype(int)

In [ ]:
'''3) reviews_per_month Al hacer referencia a cantidad de reviews mensuales,
    un missing o nulo, debería ser igual a 0.
'''
df.reviews_per_month = df.reviews_per_month.fillna(0)

In [ ]:
'''4) host_response_time se refiere a la periodicidad de respuesta del antiftrión y es de tipo texto,
    en este caso vamos a crear una columna adicional de si hay o no respuesta del anfitrion
    y mapear con valores numéricos el texto de respuesta
'''
df.host_response_time.value_counts(dropna=False)
map_response_time = {
    "within an hour": 1,
    "within a few hours": 2,
    "within a day": 3,
    "a few days or more": 4
}
df["host_response_time_num"] = df["host_response_time"].map(map_response_time)
df["host_response_time_num"] = df["host_response_time_num"].fillna(4)

df["has_host_responded"] = df["host_response_time"].isna().astype(int)

In [ ]:
'''5) has_availability se refiere a la disponibilidad del alojamiento,
    por lo que un nulo sería un valor 0.
'''
df.has_availability = df.has_availability.isna().astype(int)

In [ ]:
'''6) bathrooms_text Es una variable texto que tiene 34 valores únicos que trataremos
    más adelante para darle sentido.
    De momento imputamos a los nulos la moda
'''
df.bathrooms_text = df.bathrooms_text.fillna(df.bathrooms_text.mode()[0])

In [ ]:
'''7) host_verifications Hace referencia a la cantidad de elementos de verificación que tiene el usuario,
    por lo que podemos convertirlo en numérica
'''
df.host_verifications = df.host_verifications.fillna("[]")
df["host_verifications_list"] = df.host_verifications.apply(ast.literal_eval)
df.host_verifications = df.host_verifications_list.apply(len)

In [ ]:
'''8) host_listings_count Son las veces que el anfitrión ha listado esta propiedad.
    Imputamos la mediana
'''
df.host_listings_count = df.host_listings_count.fillna(df.host_listings_count.median())

In [ ]:
'''9) host_total_listings_count Son las veces que el anfitrión ha listado una propiedad.
    Imputamos la mediana
'''
df.host_total_listings_count = df.host_total_listings_count.fillna(df.host_total_listings_count.median())

In [ ]:
'''10) host_since Fecha de antigüedad del antifrión.
    Siguiendo la lógica de casos anteriores, vamos a convertirlo en días.
'''
df.host_since = pd.to_datetime(df.host_since, errors= "coerce")
df.host_since = (reference_date - df.host_since).dt.days
df.host_since = df.host_since.fillna(df.host_since.median())
df.host_since = df.host_since.astype(int)

In [ ]:
'''11) host_name Hace referencia a un dato personal que no tiene relevancia para el estudio,
    por lo que dropeamos la columna
'''
df.drop(columns="host_name", inplace= True)

In [ ]:
df.drop(columns=["last_review","first_review","host_response_time","host_verifications_list"], inplace= True)

### 9. Anomalías, errores y Outliers

In [ ]:
df.sample(10)

In [ ]:
# Sacamos un describe con los percentiles 1% y 99% para ver si hay mucha diferencia entre el mín y max
df.describe(percentiles=[0.01, 0.99]).T[
    ["min", "1%", "99%", "max"]
]

In [ ]:
# Aplicaremos winsorización a aquellas features cuyos valor máximo supere por mucho al 99% y no tenga lógica
# La winsorización consiste en igualar los valores máximos al 99%
columns_to_winsor = ["bathrooms","bedrooms","beds","price","reviews_per_month"]
for col in columns_to_winsor:
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=upper)

En este apartado hemos detectado una serie de outliers máximos que consideramos poco lógicos por baja probabilidad, como que un alojamiento en madrid tenga 15 baños cuando el 99% máximo tienen 4.
Otras features como **estimated_revenue_l365d** no tratamos aquí sus outliers ya que lo tratamos al crear revenue_log

### 10. Feature engineering

Aquí vamos a tratar las variables categóricas o cualitativas para darles un sentido ordinal o numérico para que el modelo pueda trabajar con ellas.


#### 10.1 Encodings

In [ ]:
# Sacaremos un listado de las columnas categóricas que tenemos actualmente en el dataframe
cols_categoricas = df.select_dtypes(include="object").columns.to_list()
cols_categoricas

In [ ]:
# Miraremos su cardinalidad para detectar binarias, baja cardinalidad o alta y ver qué hacer con cada una
cardinalidad_categoricas = pd.DataFrame({
    "col_categorica": cols_categoricas,
    "valores_unicos": [df[col].nunique() for col in cols_categoricas]
}).sort_values(by="valores_unicos", ascending=False)
cardinalidad_categoricas


Comenzamos por las features **last_scraped** y **calendar_last_scraped** que son fechas en las que se obtuvo la información del anuncio.
En este caso vemos que sus valores son los mismos y no aportan información para los modelos, por lo que las eliminamos.

In [ ]:
df[["last_scraped","calendar_last_scraped"]].value_counts()

In [ ]:
df.drop(columns=["last_scraped","calendar_last_scraped"],inplace=True)

Seguimos por aquellas de baja cardinalidad como **host_has_profile_pic,host_is_superhost,host_identity_verified,instant_bookable y source** que son susceptibles de ser binarias u objeto de hot encoding

In [ ]:
for col in ["host_has_profile_pic","host_is_superhost","host_identity_verified","instant_bookable","source"]:
    print(f"Para \033[1m{col}\033[0m existen los siguientes valores:\n"+
          f"{df[col].value_counts()}\n")

Vemos que aquellas con 3 valores únicos son booleanas y datos desconocidos que trataremos como False, por lo que aplicaremos un mapeo a cada una.<br>
La feature **source** al sólo tener un valor no aporta información para el entrenamiento por lo que descartaremos.

In [ ]:
mapeo_binarias = {
    "t": 1,
    "f": 0,
    "Unknown": 0
}

df.host_has_profile_pic = df.host_has_profile_pic.map(mapeo_binarias)
df.host_is_superhost = df.host_is_superhost.map(mapeo_binarias)
df.host_identity_verified = df.host_identity_verified.map(mapeo_binarias)
df.instant_bookable = df.instant_bookable.map(mapeo_binarias)
df.drop(columns="source",inplace=True)

Analizando las siguientes categóricas, vemos que **room_type** y **room_type_simplified**, parecen ser columnas duplicadas, por lo que vamos a ver sus valores y tomar una decisión

In [ ]:
df[["room_type", "room_type_simplified"]].value_counts()

In [ ]:
df.drop(columns="room_type_simplified", inplace= True)
df = pd.get_dummies(df,columns=["room_type"],drop_first=True)

Como vemos que tienen los mismos valores, eliminamos **room_type_simplified** y hacemos un One-Hot encoding de **room_type** de forma que nos cree columnas booleanas para los valores Private room,Shared room y Hotel room, de forma que si estas 3 son 0 significará que el apartamento es Entire home/apt

Con **bathrooms_text** ya imputamos anteriormente la moda a los nulos pero dejamos que siguiera siendo texto, ahora hay que tratarla, por lo que extraeremos el número de baños y el texto private o shared para crear una columna de si es bamo compartido o privado

In [ ]:
df["bathrooms_num"] = (
    df["bathrooms_text"]
        .str.extract(r"(\d+\.?\d*)")
        .astype(float)
)
df.loc[df["bathrooms_text"].str.contains("half", case=False, na=False) &
       df["bathrooms_num"].isna(), "bathrooms_num"] = 0.5

df["is_bathroom_shared"] = (
    df["bathrooms_text"]
        .str.contains("shared", case=False, na=False)
        .astype(int)
)
df.drop(columns="bathrooms_text",inplace=True)


Ahora pasamos a columnas de alta cardinalidad por lo que el hot-encoding puede no compensar al crear muchas columnas, empezamos con **neighbourhood_group_cleansed**

In [ ]:
df.neighbourhood_group_cleansed.value_counts()

A pesar de que un One-hot nos crearía 20 columnas adicionales, al ser un Dataset grande, no supone un problema y mantiene la información sin perderla.
Tampoco es viable un mapeo de relevancia por cada distrito ya que sería asignar de forma subjetiva la importancia a cada distrito y contaminaria el modelo, por lo que vamos a optar por One-Hot

In [ ]:
df = pd.get_dummies(df,columns=["neighbourhood_group_cleansed"],prefix="ng",drop_first=True)

Ahora nos quedan dos features de alta cardinalidad: **neighbourhood_cleansed** y **property_type**, por lo que One-Hot Encoding no es una opción viable, tendremos que aplicar un target encoding.<br>
Al ser esta una técnica que emplea datos de la variable target, no puede ser aplicada sobre todo el dataset, ya que si se aplica sobre test, aporta al modelo información de la target que no debería conocer.<br><br>
Por tanto y como transformamos nuestra target inicial **estimated_revenue_l365d** en **revenue_log** que será la nueva target,volvemos a dividir en test y train y aplicaremos estas modificaciones únicamente a train

In [ ]:
X = df.drop(columns=['revenue_log', 'estimated_revenue_l365d'])
y = df['revenue_log']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Sobreescribimos los datos del test

X_test.to_csv("X_test.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

Ahora como comentamos anteriormente al hacer target encoding para **neighbourhood_cleansed**, calcularemos la media de revenue_log para cada valor de nuestra feature en train únicamente e imputaremos la media a test para que no haya filtrado de datos

In [ ]:
mean_nh_target = X_train.join(y_train).groupby('neighbourhood_cleansed')['revenue_log'].mean()
X_train['neighbourhood_revenue'] = X_train['neighbourhood_cleansed'].map(mean_nh_target)

# Realizamos lo mismo en test sustituyendo por la media de Train

X_test['neighbourhood_revenue'] = X_test['neighbourhood_cleansed'].map(mean_nh_target)
X_test['neighbourhood_revenue'] = X_test['neighbourhood_revenue'].fillna(y_train.mean())

X_train.drop(columns="neighbourhood_cleansed",inplace=True)
X_test.drop(columns="neighbourhood_cleansed",inplace=True)

Por último la feature **property_type**, para este caso planteamos un frequency_encoding, así simplificamos el tratamiento de su variable en base a su frecuencia o "popularidad", sin embargo los valores de la feature tienen 2 categorías con muchos valores (Entire rental unit = 7870, Private room in rental unit = 2025) y el resto son < 300 <br>
Por lo que haremos igualmengte target encoding

In [ ]:
mean_pt_target = X_train.join(y_train).groupby('property_type')['revenue_log'].mean()
X_train['pt_revenue'] = X_train['property_type'].map(mean_pt_target)

X_test['pt_revenue'] = X_test['property_type'].map(mean_pt_target)
X_test['pt_revenue'] = X_test['pt_revenue'].fillna(y_train.mean())

X_train.drop(columns="property_type",inplace=True)
X_test.drop(columns="property_type",inplace=True)

### 11. Feature reduction

Como tenemos muchas columnas vamos a intentar eliminar alguna redundante o que sea poco útil para el modelo, aunque ya hemos eliminado y tratado suficiente y la mayoría estñan preparadas para el modelo

In [ ]:
print("Shape de X_train",X_train.shape)
print("Shape de X_test",X_test.shape)

In [ ]:
# Listamos las columnas
X_train.columns

In [ ]:
# Eliminamos todas aquellas redundantes de calculated_host_listings_count*
cols_to_drop = ['calculated_host_listings_count',
                'calculated_host_listings_count_entire_homes',
                'calculated_host_listings_count_private_rooms',
                'calculated_host_listings_count_shared_rooms']

In [ ]:
# Visualizamos has_availability y su distribución
X_train.has_availability.value_counts()

In [ ]:
# La incluímos en cols_to_drop al tener poco reparto de valores y aportar poca información por ello
cols_to_drop.append("has_availability")
X_train.drop(columns=cols_to_drop,inplace=True)
X_test.drop(columns=cols_to_drop,inplace=True)
print("Shape de X_train",X_train.shape)
print("Shape de X_test",X_test.shape)

### 12 Escoger métrica del modelo (Regresión)  

#### Métrica

Modelamos `revenue_log = log1p(revenue)` para reducir asimetría y el efecto de outliers típicos en variables monetarias.  
Usamos **RMSE** como métrica principal sobre `revenue_log` porque penaliza más los errores grandes y permite comparar modelos de forma consistente.  
Para interpretar en la escala original, deshacemos la transformación con `expm1` al evaluar: `revenue = expm1(revenue_log)`.

### 13–14) Protocolo de evaluación: Cross-Validation (CV)

Antes de comparar modelos, fijamos un **único protocolo de validación** para que la comparación sea justa:
- Usaremos **K-Fold CV** (k=5) sobre `X_train`/`y_train`.
- `shuffle=True` mezcla los datos antes de partirlos (útil si no hay orden temporal).
- `random_state=42` garantiza reproducibilidad.

**Regla clave:** todos los modelos se evalúan con **el mismo CV** y la misma métrica (RMSE en `revenue_log`).

In [ ]:
from sklearn.model_selection import KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)

## 14) Pipelines por modelo
Creamos **un pipeline por modelo** para evitar leakage en CV y comparar justo.  
El preprocesado es común (ej. imputación) y solo cambia el estimador.  

### Modelos a comparar
Probamos tres enfoques para cubrir distintos niveles de complejidad:
- **Ridge (L2)**: baseline lineal fuerte. La regularización L2 reduce overfitting y ayuda cuando hay colinealidad y muchas features.
- **Random Forest**: captura no linealidades e interacciones sin requerir escalado; suele ser robusto.
- **XGBoost**: boosting, normalmente muy competitivo en datos tabulares al modelar patrones complejos.

**Haremos un CV para cada modelos y luego nos quedaremos con el mejor**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

pipelines = {"Ridge": Pipeline([("imputer", SimpleImputer(strategy="median")),
                                ("scaler", StandardScaler()),
                                ("model", Ridge(alpha= 10.0))]),
                                
                                "RandomForest": Pipeline([("imputer", SimpleImputer(strategy="median")),
                                                          ("model", RandomForestRegressor(n_estimators= 400,
                                                                                          random_state= 42,
                                                                                          n_jobs= -1))]),
                                                              
                                "XGBoost": Pipeline([("imputer", SimpleImputer(strategy="median")),
                                                     ("model", XGBRegressor(n_estimators= 600,
                                                                            learning_rate= 0.05,
                                                                            max_depth= 6,
                                                                            subsample= 0.9,
                                                                            colsample_bytree= 0.9,
                                                                            reg_lambda= 1.0,
                                                                            random_state= 42,
                                                                            n_jobs= -1))])
                                                                                           }

results = {}

for name, pipe in pipelines.items():
    rmse = -cross_val_score(pipe, X_train, y_train, cv= cv, scoring= "neg_root_mean_squared_error").mean()
    results[name] = rmse

    print(f"CV RMSE (log) {name}: {rmse:.3f}")

best_name = min(results, key= results.get)
best_pipe = pipelines[best_name]

print(f"\nMejor por CV (log RMSE): {best_name} -> {results[best_name]:.3f}")

### Modelado: baseline y comparativa (con validación cruzada)

Para comparar modelos de forma justa, usamos el **mismo preprocesado** (imputación; escalado solo para modelos lineales), el mismo **K-Fold CV** y la misma métrica (**RMSE sobre `revenue_log`**).

**Baseline (referencia mínima):**
- **Ridge (L2)** actúa como baseline lineal robusto: es simple, rápido y suele rendir bien con muchas variables y posible colinealidad.

**Modelos comparados:**
- **Ridge (L2)**: referencia lineal.
- **Random Forest**: captura no linealidades e interacciones sin requerir escalado.
- **XGBoost**: boosting, normalmente muy competitivo en datos tabulares al combinar muchos árboles “débiles” con regularización.

**Resultados (CV RMSE en `revenue_log`, menor es mejor):**
- Ridge: 1.273  
- Random Forest: 0.068  
- XGBoost: 0.054  

Con estos resultados, **XGBoost** fue el mejor modelo en CV y lo seleccionamos como candidato para la fase de **optimización de hiperparámetros** y evaluación final.

> Nota: reportamos RMSE en escala logarítmica porque modelamos `revenue_log = log1p(revenue)` para reducir asimetría y el impacto de outliers.

### XGBoost: hiperparámetros elegidos (configuración base, sin tuning)
Usamos una configuración **conservadora** para obtener buen rendimiento sin optimización exhaustiva:

- `n_estimators=600` + `learning_rate=0.05`: más árboles con un paso de aprendizaje pequeño → suele generalizar mejor que pocos árboles con learning rate alto.
- `max_depth=6`: profundidad moderada para capturar no linealidades sin volver el modelo demasiado complejo.
- `subsample=0.9` y `colsample_bytree=0.9`: “submuestreo” de filas y columnas en cada árbol para reducir overfitting y mejorar robustez.
- `reg_lambda=1.0`: regularización L2 estándar para penalizar complejidad y estabilizar el modelo.
- `random_state=42`: reproducibilidad de resultados.
- `n_jobs=-1`: usa todos los núcleos disponibles para acelerar el entrenamiento.

Estos valores funcionan como **baseline fuerte**; la optimización de hiperparámetros se realizará en el siguiente bloque del proyecto.

In [ ]:
y_test_eur = np.expm1(y_test)
print("Median revenue (test):", np.median(y_test_eur))
print("Mean revenue (test):  ", np.mean(y_test_eur))

In [ ]:
y_test_eur = np.expm1(y_test)
for p in [50, 75, 90, 95, 99]:
    print(p, np.percentile(y_test_eur, p))
print("max", y_test_eur.max())

### Chequeo rápido del target (escala original)
Antes de interpretar las métricas, deshacemos `log1p` con `expm1` para ver el revenue anual en euros.  
Calculamos **media, mediana y percentiles** del revenue en test para entender su magnitud típica y su dispersión.

### Modelos a comparar
Probamos tres enfoques para cubrir distintos niveles de complejidad:
- **Ridge (L2)**: baseline lineal fuerte. La regularización L2 reduce overfitting y ayuda cuando hay colinealidad y muchas features.
- **Random Forest**: captura no linealidades e interacciones sin requerir escalado; suele ser robusto.
- **XGBoost**: boosting, normalmente muy competitivo en datos tabulares al modelar patrones complejos.

In [ ]:
from sklearn.metrics import root_mean_squared_error

XGBoost_model = best_pipe
XGBoost_model.fit(X_train, y_train)

y_pred_log = XGBoost_model.predict(X_test)

rmse_test_log = root_mean_squared_error(y_test, y_pred_log)
print(f"Test RMSE (log) XGBoost: {rmse_test_log:.3f}")

rmse_test_eur = root_mean_squared_error(np.expm1(y_test), np.expm1(y_pred_log))
print(f"Test RMSE (€)   XGBoost: {rmse_test_eur:.2f}")

### Interpretación del RMSE en euros
Aunque el **RMSE (€)** pueda parecer alto, el revenue tiene una **cola muy pesada**: existe un outlier cercano a **2M€**.  
Como el RMSE penaliza mucho los errores grandes (errores al cuadrado), unos pocos valores extremos pueden dominar la métrica en euros, aunque el modelo funcione bien para la mayoría de casos.

## 15) Optimización de hiperparámetros 

Tras construir varios modelos base (baseline), el siguiente paso consiste en mejorar su rendimiento ajustando sus hiperparámetros. La optimización busca encontrar la combinación que minimiza el error y mejora la capacidad de generalización.

Se ha utilizado:

- Pipeline de sklearn para encapsular el flujo (preprocesado + modelo)
- RandomizedSearchCV para bsucar las combinaciones de hiperparámetros de forma eficiente
- La validación cruzada (CV) para ecitar el overfitting

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
#Ridge: Modelo lineal regularizado

ridge_pipe = pipelines["Ridge"]

ridge_params = {"model__alpha" : np.logspace(-3,3,50)}

ridge_search = RandomizedSearchCV(ridge_pipe,ridge_params,n_iter=20,scoring="neg_root_mean_squared_error", cv=5, random_state=42,n_jobs=-1)

ridge_search.fit(X_train, y_train)

print("Mejores parámetros Ridge:", ridge_search.best_params_)
print("Mejores RMSE CV Ridge:", -ridge_search.best_score_)



In [ ]:
#Random Forest

rf_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(random_state=42))
])

rf_params = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]
}

rf_search = RandomizedSearchCV(
    rf_pipe,
    rf_params,
    n_iter=25,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

print("Mejores parámetros RF:", rf_search.best_params_)
print("Mejor RMSE CV RF:", -rf_search.best_score_)

In [ ]:
#XGBoost

xgb_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBRegressor(
        random_state=42,
        objective="reg:squarederror",
        verbosity=0
    ))
])

xgb_params = {
    "model__n_estimators": [200, 400, 600],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 5, 7, 10],
    "model__subsample": [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 1.0],
    "model__gamma": [0, 0.1, 0.3],
    "model__reg_alpha": [0, 0.1, 1],
    "model__reg_lambda": [1, 1.5, 2]
}

xgb_search = RandomizedSearchCV(
    xgb_pipe,
    xgb_params,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X_train, y_train)

print("Mejores parámetros XGB:", xgb_search.best_params_)
print("Mejor RMSE CV XGB:", -xgb_search.best_score_)

In [ ]:
best_pipe = xgb_search.best_estimator_

best_pipe.fit(X_train, y_train)

y_pred_log = best_pipe.predict(X_test)

rmse_test_log = root_mean_squared_error(y_test, y_pred_log)
print(f"Test RMSE (log): {rmse_test_log:.3f}")

rmse_test_eur = root_mean_squared_error(
    np.expm1(y_test),
    np.expm1(y_pred_log)
)
print(f"Test RMSE (€): {rmse_test_eur:.2f}")

La optimización de hiperparámetros se ha llevado a cabo con el objetivo de maximizar el rendimiento de los modelos desarrollados durante la fase de modelado, garantizando al mismo tiempo su capacidad de generalización.

Para ello, se ha optado por el uso de RandomizedSearchCV frente a otras alternativas como Grid Search, debido a su mayor eficiencia computacional. Este método permite explorar el espacio de hiperparámetros de forma aleatoria, evaluando un número limitado de combinaciones pero cubriendo un rango amplio de valores, lo que resulta especialmente adecuado en contextos con múltiples modelos y recursos limitados.

El uso de pipelines ha sido clave para asegurar la correcta integración del preprocesado (imputación de valores nulos y escalado de variables) dentro del proceso de validación cruzada. Esto evita fugas de información (data leakage) y garantiza que las transformaciones se apliquen de forma consistente tanto en entrenamiento como en validación.

La métrica seleccionada para la optimización ha sido el error cuadrático medio (RMSE), dado que el problema abordado es de regresión sobre variables continuas relacionadas con precios. Esta métrica penaliza en mayor medida los errores grandes, lo que resulta especialmente relevante en el contexto del análisis inmobiliario.

Finalmente, la validación cruzada con 5 particiones ha permitido obtener una estimación robusta del rendimiento del modelo, reduciendo la dependencia de una única partición de los datos y mejorando la fiabilidad de los resultados obtenidos.

## 16) Evaluación final en test

In [ ]:
from sklearn.metrics import root_mean_squared_error, r2_score

# Predicción sobre test
y_pred = best_pipe.predict(X_test)

# Métricas
rmse_test = root_mean_squared_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)

print(f"RMSE (test): {rmse_test:.2f}")
print(f"R2 (test): {r2_test:.4f}")

In [ ]:
# RMSE en euros (escala original)
rmse_test_eur = root_mean_squared_error(np.expm1(y_test), np.expm1(y_pred_log))
print(f"RMSE (test, €): {rmse_test_eur:.2f}")

La evaluación final se ha realizado sobre el conjunto de test, no utilizado durante el entrenamiento ni en la optimización de hiperparámetros. Esto garantiza una estimación realista del rendimiento del modelo en datos no vistos.

La combinación de validación cruzada, optimización de hiperparámetros y evaluación final en test ha permitido obtener un modelo robusto, capaz de generalizar correctamente y mejorar significativamente los resultados iniciales del baseline.

## 17) Persistencia del modelo

In [ ]:
import joblib

joblib.dump(best_pipe, "modelo_optimizado.pkl")

In [ ]:
modelo_cargado = joblib.load("modelo_optimizado.pkl")

predicciones_modelo = modelo_cargado.predict(X_test)

El modelo final optimizado ha sido persistido mediante la utilización de joblib lo que permite su reutilización en entornos productivos y análisis posteriores sin necesidad de reentrenamiento. Esto es fundamental para garantizar la reproducibilidad del proyecto.